In [2]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
print(tf.__version__)

import keras
print(keras.__version__)

2.20.0
3.11.3


In [3]:
import os
import sys

# Get the path to the project root directory (the parent of 'notebooks')
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add the project root to the Python path
if project_root not in sys.path:
    sys.path.append(project_root)

# Now you can import your code from the src directory
from src.main import main

In [4]:
import sys

# Replace this with the actual command-line arguments you want to pass.
# The first element is the script name.
simulated_args = ['main.py', 'play_games', '--count', '1000']

# Manually set sys.argv to your list of arguments.
sys.argv = simulated_args

# Now, call your main function.
# Your argument parser will read from the sys.argv list you just created.
main()

In [ ]:
from src.logger import *
from training_data import prepare_training_data

reader = read_for_player('/app/data/game_log.avro', 'alice')
mydata = prepare_training_data('/app/data/game_log.avro', 'alice')

x_train = tf.stack([x[3] for x in mydata])
y_train = tf.stack([[x[0],x[1],x[2]] for x in mydata])
m = x_train.shape[0]
y_pred = np.zeros(m)


In [ ]:
import importlib
import src.training_data as t
import src.main as m
importlib.reload(m)

In [ ]:
import tensorflow as tf
from tensorflow import keras

num_inputs = 330
hidden_units = 50

# Create the Sequential model
model = keras.models.Sequential()

# Add the explicit Input layer as the first layer
model.add(keras.Input(shape=(num_inputs,)))

# Add the hidden layers as before, but without the input_shape argument
model.add(keras.layers.Dense(units=hidden_units, activation='relu'))
model.add(keras.layers.Dense(units=hidden_units, activation='relu'))

# Add the output layer
model.add(keras.layers.Dense(units=1, activation='linear'))

# Print a summary of the model's architecture
model.summary()

In [ ]:
import tensorflow as tf
from tensorflow import keras

class GameStateWeightedLoss(tf.keras.losses.Loss):
    """
    Custom loss function that directly guides the gradient based on game outcome.

    The `y_true` tensor is expected to have a shape of (batch_size, 3),
    with columns for [true_score, move_number, total_moves].
    """
    def __init__(self, name="game_state_weighted_loss"):
        super().__init__(name=name)

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, dtype=tf.float32)
        y_pred = tf.cast(y_pred, dtype=tf.float32)

        y_true_score = y_true[:, 0]
        move_number = y_true[:, 1]
        total_moves = y_true[:, 2]

        loss =  y_true_score * tf.math.divide_no_nan(move_number, total_moves)
        return tf.reduce_mean(loss)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import regularizers
import numpy as np

# A small value for the L2 regularization factor. This is a hyperparameter to tune.
l2_factor = 0.001

# Define the model architecture.
# The L2 regularization is applied to the 'kernel' (weights) of the hidden layers.
model = keras.models.Sequential([
    keras.Input(shape=(330,)),
    keras.layers.Dense(50, activation='relu', kernel_regularizer=regularizers.l2(l2_factor)),
    keras.layers.Dense(50, activation='relu', kernel_regularizer=regularizers.l2(l2_factor)),
    # The output layer typically does not use regularization.
    keras.layers.Dense(1, activation='linear')
])

# The optimizer controls the learning process. 'adam' is a good default choice.
# The loss function will be provided separately as you noted.
model.compile(optimizer='adam', loss=GameStateWeightedLoss())

# --- The Training Process ---
# This is where backpropagation happens.
# The model uses the compiled loss function and optimizer to update weights.
model.fit(x=x_train, y=y_train, epochs=10, batch_size=32)

model.summary()


In [ ]:
from logger import 

In [ ]:
from random import Random

from dice_source import DiceSource

r = Random(443)
temperature = 2
alice = NNPlayer('alice', model, r, temperature)
bob = RandomPlayer('b', r)
dice_source = DiceSource(r)
game_manager = GameManager(alice, bob, dice_source)
r = [game_manager.run_once() for _ in range(100)]
print(r)




                 

In [ ]:
def create_and_save_model():
    # Create a simple Sequential model
    model = keras.Sequential([
        keras.layers.Dense(10, activation='relu', input_shape=(4,)),
        keras.layers.Dense(1, activation='linear')
    ])
    
    # Compile the model
    model.compile(optimizer='adam', loss='mean_squared_error')
    
    # Generate some dummy data for training
    x_train = np.random.rand(100, 4)
    y_train = np.random.rand(100, 1)

    # Train the model
    model.fit(x_train, y_train, epochs=10, verbose=0)
    
    # Save the model
    model.save('my_simple_model.keras')
    print("Model created and saved to 'my_simple_model.keras'")

# Run the function to create and save the model for this example
create_and_save_model()

# Now, let's load the model back
loaded_model = keras.models.load_model('my_simple_model.keras')
print("\nModel loaded successfully!")

# Let's create some new input data to make a prediction on.
# It must have the same shape as the training data, in this case, a 2D array.
new_data = np.array([[0.5, 0.2, 0.8, 0.1]])

# Call the predict method
prediction = loaded_model.predict(new_data)

In [6]:
import timeit
import numpy as np

# Create a sample data set
size = 1_000_000
python_list = list(range(size))
numpy_array = np.arange(size)

# Time the list copy
list_copy_time = timeit.timeit('list_copy = list(original_list)', globals={'original_list': python_list}, number=100)

# Time the numpy array copy
numpy_copy_time = timeit.timeit('numpy_copy = original_array.copy()', globals={'original_array': numpy_array}, number=100)

print(f"List copy time: {list_copy_time:.6f} seconds")
print(f"NumPy copy time: {numpy_copy_time:.6f} seconds")
print(f"NumPy is {list_copy_time / numpy_copy_time:.2f}x faster.")

List copy time: 0.559161 seconds
NumPy copy time: 0.081210 seconds
NumPy is 6.89x faster.


In [7]:
import timeit
import numpy as np

size = 100_000
list1 = list(range(size))
list2 = list(range(size))
arr1 = np.arange(size)
arr2 = np.arange(size)

list_check_time = timeit.timeit('list1 == list2', globals=globals(), number=100)
numpy_check_time = timeit.timeit('np.all(arr1 == arr2)', globals=globals(), number=100)

print(f"List equality check time: {list_check_time:.6f} seconds")
print(f"NumPy equality check time: {numpy_check_time:.6f} seconds")
print(f"NumPy is {list_check_time / numpy_check_time:.2f}x faster.")

List equality check time: 0.049666 seconds
NumPy equality check time: 0.003322 seconds
NumPy is 14.95x faster.
